In [ ]:
# This is a version of aves_classifier.ipynb. I am running it in Google Colab
# so that I can use its GPU.

import librosa
import numpy as np
import pandas as pd
import os
import math
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Imports the files I made in Visual Studio Code.

import glob
import re

def chunk_num(path):
    return int(re.search(r'_(\d+)\.npy$', path).group(1))

x_files = sorted(glob.glob("/content/X_*.npy"), key=chunk_num)
y_files = sorted(glob.glob("/content/y_*.npy"), key=chunk_num)

print(len(x_files), len(y_files))
print(x_files[:5])
print(y_files[:5])

54 54
['/content/X_0.npy', '/content/X_1000.npy', '/content/X_2000.npy', '/content/X_3000.npy', '/content/X_4000.npy']
['/content/y_0.npy', '/content/y_1000.npy', '/content/y_2000.npy', '/content/y_3000.npy', '/content/y_4000.npy']


In [ ]:
# Confirms alignment between x and y files.

for xf, yf in zip(x_files[:10], y_files[:10]):
    print(xf, yf)

/content/X_0.npy /content/y_0.npy
/content/X_1000.npy /content/y_1000.npy
/content/X_2000.npy /content/y_2000.npy
/content/X_3000.npy /content/y_3000.npy
/content/X_4000.npy /content/y_4000.npy
/content/X_5000.npy /content/y_5000.npy
/content/X_6000.npy /content/y_6000.npy
/content/X_7000.npy /content/y_7000.npy
/content/X_8000.npy /content/y_8000.npy
/content/X_9000.npy /content/y_9000.npy


In [ ]:
for xf, yf in zip(x_files[:5], y_files[:5]):
    Xc = np.load(xf)
    yc = np.load(yf)
    print(xf, len(Xc), len(yc))

/content/X_0.npy 1000 1000
/content/X_1000.npy 1000 1000
/content/X_2000.npy 1000 1000
/content/X_3000.npy 1000 1000
/content/X_4000.npy 1000 1000


In [ ]:
# Creates training and validation sets.

train_x_files, val_x_files, train_y_files, val_y_files = train_test_split(
    x_files,
    y_files,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Performs a train-test-validation split.
# Ensures that every class has enough data.

NUM_CLASSES = 112

def load_chunk(x_path, y_path):
    x_path = x_path.decode("utf-8")
    y_path = y_path.decode("utf-8")

    X_chunk = np.load(x_path).astype(np.float32)
    y_chunk = np.load(y_path).astype(np.float32)

    return X_chunk, y_chunk

# We convert X and y to tensorflow to reduce memory requirements.

def tf_load_chunk(x_path, y_path):
    X_chunk, y_chunk = tf.numpy_function(
        load_chunk,
        [x_path, y_path],
        [tf.float32, tf.float32] # Convert back to float for stability
    )

    X_chunk.set_shape([None, 128, 313, 1])
    y_chunk.set_shape([None, NUM_CLASSES])

    return X_chunk, y_chunk

# num_parallel_calls is reduced because running too many things in parallel
# crashes the code. prefetch is there for the same reason.

In [ ]:
batch_size = 64

train_samples = sum(np.load(f).shape[0] for f in train_y_files)
val_samples   = sum(np.load(f).shape[0] for f in val_y_files)

steps_per_epoch = math.ceil(train_samples / batch_size)
validation_steps = math.ceil(val_samples / batch_size)

print(train_samples, val_samples)
print(steps_per_epoch, validation_steps)

42412 11000
663 172


In [ ]:
# SpecAugment introduces some light noise to make the model more robust.
# The model already fits well without SpecAugment, so this will be mild.

def spec_augment_tf(x, y, p=0.15, max_time_mask=10, max_freq_mask=5):
    def augment():
        # x shape: (128, frames, 1)
        n_mels = tf.shape(x)[0]
        n_frames = tf.shape(x)[1]

        x_aug = tf.identity(x)

        # time mask
        t = tf.random.uniform([], 0, max_time_mask, dtype=tf.int32)
        t0 = tf.random.uniform([], 0, tf.maximum(1, n_frames - t), dtype=tf.int32)

        time_mask = tf.concat([
            tf.ones((n_mels, t0, 1)),
            tf.zeros((n_mels, t, 1)),
            tf.ones((n_mels, n_frames - t0 - t, 1))
        ], axis=1)

        x_aug = x_aug * time_mask

        # frequency mask
        f = tf.random.uniform([], 0, max_freq_mask, dtype=tf.int32)
        f0 = tf.random.uniform([], 0, tf.maximum(1, n_mels - f), dtype=tf.int32)

        freq_mask = tf.concat([
            tf.ones((f0, n_frames, 1)),
            tf.zeros((f, n_frames, 1)),
            tf.ones((n_mels - f0 - f, n_frames, 1))
        ], axis=0)

        x_aug = x_aug * freq_mask

        return x_aug, y

    return tf.cond(
        tf.random.uniform([]) < p,
        augment,
        lambda: (x, y)
    )
train_ds = tf.data.Dataset.from_tensor_slices((train_x_files, train_y_files))
train_ds = train_ds.shuffle(len(train_x_files))
train_ds = train_ds.map(tf_load_chunk, num_parallel_calls=2)
train_ds = train_ds.unbatch()

# ✅ Apply SpecAugment HERE
train_ds = train_ds.map(spec_augment_tf, num_parallel_calls=2)

train_ds = train_ds.shuffle(2000)
train_ds = train_ds.batch(batch_size)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
train_ds = train_ds.repeat()

# The validation set does NOT use SpecAugment.
val_ds = tf.data.Dataset.from_tensor_slices((val_x_files, val_y_files))
val_ds = val_ds.map(tf_load_chunk, num_parallel_calls=2)
val_ds = val_ds.unbatch()
val_ds = val_ds.batch(batch_size)
val_ds = val_ds.prefetch(1)

In [ ]:
# We use a fairly large CNN because we have a lot of data and many classes.
# Regularizers ameliorate overfitting.
# They penalize overly complicated layers and large parameters.

input_shape = (128, 313, 1)

NUM_CLASSES = 112

inputs = layers.Input(shape=input_shape)

x = inputs

for filters in [64, 128, 256, 256]:
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)

# Hybrid pooling
x_avg = layers.GlobalAveragePooling2D()(x)
x_max = layers.GlobalMaxPooling2D()(x)
x = layers.Concatenate()([x_avg, x_max])

x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(NUM_CLASSES, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(curve='ROC', multi_label=True, name='roc_auc'),
        tf.keras.metrics.AUC(curve='PR', multi_label=True, name='pr_auc')
    ]
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_pr_auc',
        patience=8,
        min_delta=0.00001,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    # This block reduces the learning rate near the solution
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        cooldown=1,
        mode='min',
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_pr_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Checks for x files that were not properly uploaded.

bad_files = []

for f in x_files:
    try:
        arr = np.load(f)
        if arr.shape[1:] != (128, 313, 1):
            print("Bad shape:", f, arr.shape)
            bad_files.append(f)
    except Exception as e:
        print("Could not load:", f)
        print(e)
        bad_files.append(f)

print("Number of bad files:", len(bad_files))
print(bad_files[:10])

Number of bad files: 0
[]


In [ ]:
# This program is too computationally intensive to run on a CPU.
# I ran it in Google Colab.
# Send ChatGPT final val_pr_auc and y_pred.std() when this is done.

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks
)

Epoch 1/40
663/663 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 0.2867 - pr_auc: 0.0095 - roc_auc: 0.5001
Epoch 1: val_pr_auc improved from None to 0.01141, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
663/663 ━━━━━━━━━━━━━━━━━━━━ 41s 48ms/step - loss: 0.1309 - pr_auc: 0.0093 - roc_auc: 0.5023 - val_loss: 0.0587 - val_pr_auc: 0.0114 - val_roc_auc: 0.5069 - learning_rate: 1.0000e-04
Epoch 2/40
663/663 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0595 - pr_auc: 0.0112 - roc_auc: 0.5107
Epoch 2: val_pr_auc improved from 0.01141 to 0.01216, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
663/663 ━━━━━━━━━━━━━━━━━━━━ 23s 35ms/step - loss: 0.0583 - pr_auc: 0.0107 - roc_auc: 0.5117 - val_loss: 0.0595 - val_pr_auc: 0.0122 - val_roc_auc: 0.5082 - learning_rate: 1.0000e-04
Epoch 3/40
663/663 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.0560 - pr_auc: 0.0115 - roc_auc: 0.5123
Epoch 3: val_pr_auc improved from 0.01216 to 0.013

In [ ]:
model.save("best_common_bird_model_gpu.keras")

In [ ]:
for xf, yf in zip(x_files[:10], y_files[:10]):
    print(xf, yf)

/content/X_0.npy /content/y_0.npy
/content/X_1000.npy /content/y_1000.npy
/content/X_10000.npy /content/y_10000.npy
/content/X_11000.npy /content/y_11000.npy
/content/X_12000.npy /content/y_12000.npy
/content/X_13000.npy /content/y_13000.npy
/content/X_14000.npy /content/y_14000.npy
/content/X_15000.npy /content/y_15000.npy
/content/X_16000.npy /content/y_16000.npy
/content/X_17000.npy /content/y_17000.npy


In [ ]:
for xf, yf in zip(x_files, y_files):
    Xc = np.load(xf)
    yc = np.load(yf)

    if len(Xc) != len(yc):
        print("Mismatch:", xf, yf, Xc.shape, yc.shape)

In [ ]:
val_counts = np.zeros(NUM_CLASSES)

for yf in val_y_files:
    val_counts += np.load(yf).sum(axis=0)

print("Zero classes in val:", np.sum(val_counts == 0))
print("Min:", val_counts.min(), "Median:", np.median(val_counts), "Max:", val_counts.max())

Zero classes in val: 0
Min: 28.0 Median: 79.0 Max: 196.0


In [ ]:
for xf, yf in zip(x_files[:5], y_files[:5]):
    print(xf, yf)
    Xc = np.load(xf)
    yc = np.load(yf)
    print(Xc.shape, yc.shape, yc.sum(axis=0).sum())

/content/X_0.npy /content/y_0.npy
(1000, 128, 313, 1) (1000, 112) 1005.0
/content/X_1000.npy /content/y_1000.npy
(1000, 128, 313, 1) (1000, 112) 1004.0
/content/X_10000.npy /content/y_10000.npy
(1000, 128, 313, 1) (1000, 112) 1002.0
/content/X_11000.npy /content/y_11000.npy
(1000, 128, 313, 1) (1000, 112) 1005.0
/content/X_12000.npy /content/y_12000.npy
(1000, 128, 313, 1) (1000, 112) 1005.0


In [ ]:
train_counts = np.zeros(NUM_CLASSES)
val_counts = np.zeros(NUM_CLASSES)

for f in train_y_files:
    train_counts += np.load(f).sum(axis=0)

for f in val_y_files:
    val_counts += np.load(f).sum(axis=0)

print("Train zero classes:", np.sum(train_counts == 0))
print("Val zero classes:", np.sum(val_counts == 0))

print("Train min/median/max:", train_counts.min(), np.median(train_counts), train_counts.max())
print("Val min/median/max:", val_counts.min(), np.median(val_counts), val_counts.max())

Train zero classes: 0
Val zero classes: 0
Train min/median/max: 153.0 315.0 846.0
Val min/median/max: 34.0 84.5 206.0
